# Sentiment annotations dataset: Merging s- and u-level sentiment annotation datasets

Merging both sentence-level annotation results (produced within ParlaSent) and the u-level sentiment annotations from the initial campaigns (Datasets/ParlaMint-SI_Annotations.csv/jsonl) and splitting the dataset into development (```Dataset/Development_data.csv/jsonl```) and test dataset (```Datasets/Test_data.csv/jsonl```).

In [1]:
import pandas as pd
import csv
import re

In [2]:
csv_data = pd.read_csv('../../Utterances/Datasets/Annotations.csv', encoding='utf-8')

In [3]:
def process_chunk(df_chunk):
    print(df_chunk.head()) 
def parse_jsonl_in_chunks(file_path, chunksize=1000):
    df_list = []  
    for df_chunk in pd.read_json(file_path, lines=True, chunksize=chunksize):
        df_list.append(df_chunk)

    full_df = pd.concat(df_list, ignore_index=True)
    return full_df


file_path = '../../ParlaSent-SI/ParlaMint-SI.meta.jsonl'
df = parse_jsonl_in_chunks(file_path, chunksize=1000)

df.head()


,sent_id,text,logit,newdoc id,text_en,metadata
0,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,Spoštovane kolegice poslanke in kolegi poslanc...,4.052144,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,"Dear fellow Members and fellow Members, ladies...",{'Text_ID': 'ParlaMint-SI-en_2022-04-06-SDZ8-I...
1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,Začenjam z nadaljevanjem 99. izredne seje Drža...,3.223034,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,I'm starting with the 99th Extraordinary Meeti...,{'Text_ID': 'ParlaMint-SI-en_2022-04-06-SDZ8-I...
2,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,"Obveščen sem, da se današnje seje ne morejo ud...",1.778516,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,I am informed that today's meeting cannot be a...,{'Text_ID': 'ParlaMint-SI-en_2022-04-06-SDZ8-I...
3,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,Vse prisotne lepo pozdravljam!,4.814360,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,I say hello to everyone who's here!,{'Text_ID': 'ParlaMint-SI-en_2022-04-06-SDZ8-I...
4,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,Prehajamo na 10.,3.219104,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,We're moving on to 10.,{'Text_ID': 'ParlaMint-SI-en_2022-04-06-SDZ8-I...


In [4]:
len(df)

3876183

In [5]:
csv_data = csv_data.drop('Unnamed: 0', axis=1)
csv_data['ID'] = csv_data['ID'].str.replace(r'(\.[^\.]+)$', r'.ana\1', regex=True)
csv_data.head()

,ID,text,tag_tamara,tag_katja,comments_tamara,comments_katja,flagged_tamara,flagged_katja,sent_tamara,sent_katja,chair,final_tag
0,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,"Ni replike na repliko. Gospa ministrica, potem...",N_Neutral,N_Neutral,"Procedural, but he is denying her the ability ...","Procedural, however still has relatively stron...",NaN,NaN,Neutral,Neutral,True,N_Neutral
1,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.u107,Hvala lepa. Besedo ima Silva Črnugelj. Priprav...,P_Neutral,P_Neutral,Procedural.,NaN,NaN,NaN,Neutral,Neutral,True,P_Neutral
2,ParlaMint-SI_2011-06-21-SDZ5-Redna-29.ana.u178,"Hvala lepa. V bistvu se strinjam z vami, gospo...",Positive,Positive,Speaker is stating positive changes and support.,"Speech seems to be fairly positive, as the spe...",NaN,NaN,Positive,Positive,False,Positive
3,ParlaMint-SI_2020-09-21-SDZ8-Redna-20.ana.u276,"Hvala, podpredsednik. Hvala tudi za vprašanje,...",Negative,Negative,"Negative opinion of the topic, the work done, ...",Could also be Mixed sentiment due to the last ...,NaN,1.0,Negative,Negative,False,Negative
4,ParlaMint-SI_2009-11-18-SDZ5-Redna-11.ana.u120,Besedo ima gospod Silven Majhenič.,P_Neutral,P_Neutral,Procedural,NaN,NaN,NaN,Neutral,Neutral,True,P_Neutral


In [6]:
order = ['ID', 'sent_id', 'text', 'text_en', 'annotations', 'metadata']
json_data = df.rename(columns={'newdoc id': 'ID', 'logit': 'annotations'})

In [7]:
json_data = json_data[order]
json_data.head()

,ID,sent_id,text,text_en,annotations,metadata
0,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,Spoštovane kolegice poslanke in kolegi poslanc...,"Dear fellow Members and fellow Members, ladies...",4.052144,{'Text_ID': 'ParlaMint-SI-en_2022-04-06-SDZ8-I...
1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,Začenjam z nadaljevanjem 99. izredne seje Drža...,I'm starting with the 99th Extraordinary Meeti...,3.223034,{'Text_ID': 'ParlaMint-SI-en_2022-04-06-SDZ8-I...
2,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,"Obveščen sem, da se današnje seje ne morejo ud...",I am informed that today's meeting cannot be a...,1.778516,{'Text_ID': 'ParlaMint-SI-en_2022-04-06-SDZ8-I...
3,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,Vse prisotne lepo pozdravljam!,I say hello to everyone who's here!,4.814360,{'Text_ID': 'ParlaMint-SI-en_2022-04-06-SDZ8-I...
4,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.u1,ParlaMint-SI_2022-04-06-SDZ8-Izredna-99.ana.se...,Prehajamo na 10.,We're moving on to 10.,3.219104,{'Text_ID': 'ParlaMint-SI-en_2022-04-06-SDZ8-I...


In [8]:
#json_data['ID'] = json_data['ID'].str.replace('.ana', '', regex=False)
#json_data['sent_id'] = json_data['sent_id'].str.replace('.ana', '', regex=False)


In [9]:
merge = pd.merge(csv_data, json_data, on='ID')
merge

,ID,text_x,tag_tamara,tag_katja,comments_tamara,comments_katja,flagged_tamara,flagged_katja,sent_tamara,sent_katja,chair,final_tag,sent_id,text_y,text_en,annotations,metadata
0,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,"Ni replike na repliko. Gospa ministrica, potem...",N_Neutral,N_Neutral,"Procedural, but he is denying her the ability ...","Procedural, however still has relatively stron...",NaN,NaN,Neutral,Neutral,True,N_Neutral,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,Ni replike na repliko.,There's no replica to the replica.,2.240412,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...
1,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,"Ni replike na repliko. Gospa ministrica, potem...",N_Neutral,N_Neutral,"Procedural, but he is denying her the ability ...","Procedural, however still has relatively stron...",NaN,NaN,Neutral,Neutral,True,N_Neutral,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Gospa ministrica, potem se boste lahko javili,...","Madam Minister, you'll be able to come forward...",2.636969,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...
2,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,"Ni replike na repliko. Gospa ministrica, potem...",N_Neutral,N_Neutral,"Procedural, but he is denying her the ability ...","Procedural, however still has relatively stron...",NaN,NaN,Neutral,Neutral,True,N_Neutral,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,Besedo pa ima gospod Gorenak.,Mr. Gorenak has the floor.,3.639261,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...
3,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.u107,Hvala lepa. Besedo ima Silva Črnugelj. Priprav...,P_Neutral,P_Neutral,Procedural.,NaN,NaN,NaN,Neutral,Neutral,True,P_Neutral,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.seg3...,Hvala lepa.,Thank you very much.,4.084590,{'Text_ID': 'ParlaMint-SI-en_2011-06-16-SDZ5-R...
4,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.u107,Hvala lepa. Besedo ima Silva Črnugelj. Priprav...,P_Neutral,P_Neutral,Procedural.,NaN,NaN,NaN,Neutral,Neutral,True,P_Neutral,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.seg3...,Besedo ima Silva Črnugelj.,"The word is ""silva blackugelj.""",3.340636,{'Text_ID': 'ParlaMint-SI-en_2011-06-16-SDZ5-R...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11760,ParlaMint-SI_2009-03-27-SDZ5-Redna-04.ana.u199,Hvala lepa. Besedo ima mag. Branko Grims. Prosim.,P_Neutral,P_Neutral,Procedural,NaN,NaN,NaN,Neutral,Neutral,True,P_Neutral,ParlaMint-SI_2009-03-27-SDZ5-Redna-04.ana.seg4...,Hvala lepa.,Thank you very much.,4.084590,{'Text_ID': 'ParlaMint-SI-en_2009-03-27-SDZ5-R...
11761,ParlaMint-SI_2009-03-27-SDZ5-Redna-04.ana.u199,Hvala lepa. Besedo ima mag. Branko Grims. Prosim.,P_Neutral,P_Neutral,Procedural,NaN,NaN,NaN,Neutral,Neutral,True,P_Neutral,ParlaMint-SI_2009-03-27-SDZ5-Redna-04.ana.seg4...,Besedo ima mag. Branko Grims.,The word has magic. Branko Grimes.,3.854102,{'Text_ID': 'ParlaMint-SI-en_2009-03-27-SDZ5-R...
11762,ParlaMint-SI_2009-03-27-SDZ5-Redna-04.ana.u199,Hvala lepa. Besedo ima mag. Branko Grims. Prosim.,P_Neutral,P_Neutral,Procedural,NaN,NaN,NaN,Neutral,Neutral,True,P_Neutral,ParlaMint-SI_2009-03-27-SDZ5-Redna-04.ana.seg4...,Prosim.,Please.,3.679140,{'Text_ID': 'ParlaMint-SI-en_2009-03-27-SDZ5-R...
11763,ParlaMint-SI_2022-03-21-SDZ8-Redna-30.ana.u17,Hvala lepa. Zadnji bo stališče Poslanske skupi...,P_Neutral,P_Neutral,Procedural,NaN,NaN,NaN,Neutral,Neutral,True,P_Neutral,ParlaMint-SI_2022-03-21-SDZ8-Redna-30.ana.seg63.1,Hvala lepa.,Thank you very much.,4.084590,{'Text_ID': 'ParlaMint-SI-en_2022-03-21-SDZ8-R...


In [10]:
merge = merge.rename(columns={'text_x':'text_utt', 'text_y':'text_sent', 'annotations':'annotation_sent', 'final_tag':'annotation_utt'})
merge = merge.drop(['tag_tamara', 'tag_katja', 'comments_tamara', 'comments_katja', 'flagged_tamara', 'flagged_katja', 'sent_tamara', 'sent_katja'], axis=1)

In [11]:
ordering = ['ID', 'sent_id', 'text_utt', 'text_sent', 'text_en', 'chair', 'annotation_utt', 'annotation_sent', 'metadata']
dataset = merge[ordering]

In [12]:
dataset

,ID,sent_id,text_utt,text_sent,text_en,chair,annotation_utt,annotation_sent,metadata
0,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...",Ni replike na repliko.,There's no replica to the replica.,True,N_Neutral,2.240412,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...
1,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...","Gospa ministrica, potem se boste lahko javili,...","Madam Minister, you'll be able to come forward...",True,N_Neutral,2.636969,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...
2,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...",Besedo pa ima gospod Gorenak.,Mr. Gorenak has the floor.,True,N_Neutral,3.639261,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...
3,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.u107,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.seg3...,Hvala lepa. Besedo ima Silva Črnugelj. Priprav...,Hvala lepa.,Thank you very much.,True,P_Neutral,4.084590,{'Text_ID': 'ParlaMint-SI-en_2011-06-16-SDZ5-R...
4,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.u107,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.seg3...,Hvala lepa. Besedo ima Silva Črnugelj. Priprav...,Besedo ima Silva Črnugelj.,"The word is ""silva blackugelj.""",True,P_Neutral,3.340636,{'Text_ID': 'ParlaMint-SI-en_2011-06-16-SDZ5-R...
...,...,...,...,...,...,...,...,...,...
11760,ParlaMint-SI_2009-03-27-SDZ5-Redna-04.ana.u199,ParlaMint-SI_2009-03-27-SDZ5-Redna-04.ana.seg4...,Hvala lepa. Besedo ima mag. Branko Grims. Prosim.,Hvala lepa.,Thank you very much.,True,P_Neutral,4.084590,{'Text_ID': 'ParlaMint-SI-en_2009-03-27-SDZ5-R...
11761,ParlaMint-SI_2009-03-27-SDZ5-Redna-04.ana.u199,ParlaMint-SI_2009-03-27-SDZ5-Redna-04.ana.seg4...,Hvala lepa. Besedo ima mag. Branko Grims. Prosim.,Besedo ima mag. Branko Grims.,The word has magic. Branko Grimes.,True,P_Neutral,3.854102,{'Text_ID': 'ParlaMint-SI-en_2009-03-27-SDZ5-R...
11762,ParlaMint-SI_2009-03-27-SDZ5-Redna-04.ana.u199,ParlaMint-SI_2009-03-27-SDZ5-Redna-04.ana.seg4...,Hvala lepa. Besedo ima mag. Branko Grims. Prosim.,Prosim.,Please.,True,P_Neutral,3.679140,{'Text_ID': 'ParlaMint-SI-en_2009-03-27-SDZ5-R...
11763,ParlaMint-SI_2022-03-21-SDZ8-Redna-30.ana.u17,ParlaMint-SI_2022-03-21-SDZ8-Redna-30.ana.seg63.1,Hvala lepa. Zadnji bo stališče Poslanske skupi...,Hvala lepa.,Thank you very much.,True,P_Neutral,4.084590,{'Text_ID': 'ParlaMint-SI-en_2022-03-21-SDZ8-R...


In [13]:
label_mapping = {
    "Negative": 0.0,
    "M_Negative": 1.0,
    "N_Neutral": 2.0,
    "P_Neutral": 3.0,
    "M_Positive": 4.0,
    "Positive": 5.0
}

dataset['annotation_utt_score'] = dataset['annotation_utt'].map(label_mapping)
dataset.head()


/var/folders/l_/t5l44v1s7d9168_x8bm5ly0h0000gn/T/ipykernel_1607/2979092426.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset['annotation_utt_score'] = dataset['annotation_utt'].map(label_mapping)


,ID,sent_id,text_utt,text_sent,text_en,chair,annotation_utt,annotation_sent,metadata,annotation_utt_score
0,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...",Ni replike na repliko.,There's no replica to the replica.,True,N_Neutral,2.240412,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...,2.0
1,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...","Gospa ministrica, potem se boste lahko javili,...","Madam Minister, you'll be able to come forward...",True,N_Neutral,2.636969,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...,2.0
2,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...",Besedo pa ima gospod Gorenak.,Mr. Gorenak has the floor.,True,N_Neutral,3.639261,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...,2.0
3,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.u107,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.seg3...,Hvala lepa. Besedo ima Silva Črnugelj. Priprav...,Hvala lepa.,Thank you very much.,True,P_Neutral,4.084590,{'Text_ID': 'ParlaMint-SI-en_2011-06-16-SDZ5-R...,3.0
4,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.u107,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.seg3...,Hvala lepa. Besedo ima Silva Črnugelj. Priprav...,Besedo ima Silva Črnugelj.,"The word is ""silva blackugelj.""",True,P_Neutral,3.340636,{'Text_ID': 'ParlaMint-SI-en_2011-06-16-SDZ5-R...,3.0


In [14]:
ord = ['ID', 'sent_id', 'text_utt', 'text_sent', 'text_en', 'chair', 'annotation_utt', 'annotation_utt_score', 'annotation_sent','metadata']
data = dataset[ord]
data.head()

,ID,sent_id,text_utt,text_sent,text_en,chair,annotation_utt,annotation_utt_score,annotation_sent,metadata
0,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...",Ni replike na repliko.,There's no replica to the replica.,True,N_Neutral,2.0,2.240412,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...
1,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...","Gospa ministrica, potem se boste lahko javili,...","Madam Minister, you'll be able to come forward...",True,N_Neutral,2.0,2.636969,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...
2,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...",Besedo pa ima gospod Gorenak.,Mr. Gorenak has the floor.,True,N_Neutral,2.0,3.639261,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...
3,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.u107,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.seg3...,Hvala lepa. Besedo ima Silva Črnugelj. Priprav...,Hvala lepa.,Thank you very much.,True,P_Neutral,3.0,4.084590,{'Text_ID': 'ParlaMint-SI-en_2011-06-16-SDZ5-R...
4,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.u107,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.seg3...,Hvala lepa. Besedo ima Silva Črnugelj. Priprav...,Besedo ima Silva Črnugelj.,"The word is ""silva blackugelj.""",True,P_Neutral,3.0,3.340636,{'Text_ID': 'ParlaMint-SI-en_2011-06-16-SDZ5-R...


In [15]:
data.to_csv('../Datasets/ParlaMint-SI_Annotations.csv', encoding='utf-8', index=False)

with open('../Datasets/ParlaMint-SI_Annotations.jsonl', 'w', encoding='utf-8') as f:
    merge.to_json(f, orient='records', lines=True, force_ascii=False)

In [16]:
from sklearn.model_selection import train_test_split
len(data)

11765

In [17]:
utterance_ids = data['ID'].unique()
len(utterance_ids)

1000

In [18]:
dev_ids, test_ids = train_test_split(utterance_ids, test_size=0.2, random_state=42)

In [19]:
len(test_ids)
len(dev_ids)

800

In [20]:
dev_data = data[data['ID'].isin(dev_ids)]
test_data = data[data['ID'].isin(test_ids)]

In [21]:
print("Dev set size:", dev_data.shape)
print("Test set size:", test_data.shape)

Dev set size: (9068, 10)
Test set size: (2697, 10)


In [22]:
dev_data.to_csv('../Datasets/Development_data.csv', index=False)
test_data.to_csv('../Datasets/Test_data.csv', index=False)

In [23]:
dev_data.head()

,ID,sent_id,text_utt,text_sent,text_en,chair,annotation_utt,annotation_utt_score,annotation_sent,metadata
0,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...",Ni replike na repliko.,There's no replica to the replica.,True,N_Neutral,2.0,2.240412,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...
1,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...","Gospa ministrica, potem se boste lahko javili,...","Madam Minister, you'll be able to come forward...",True,N_Neutral,2.0,2.636969,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...
2,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.u90,ParlaMint-SI_2010-09-07-SDZ5-Izredna-28.ana.se...,"Ni replike na repliko. Gospa ministrica, potem...",Besedo pa ima gospod Gorenak.,Mr. Gorenak has the floor.,True,N_Neutral,2.0,3.639261,{'Text_ID': 'ParlaMint-SI-en_2010-09-07-SDZ5-I...
3,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.u107,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.seg3...,Hvala lepa. Besedo ima Silva Črnugelj. Priprav...,Hvala lepa.,Thank you very much.,True,P_Neutral,3.0,4.084590,{'Text_ID': 'ParlaMint-SI-en_2011-06-16-SDZ5-R...
4,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.u107,ParlaMint-SI_2011-06-16-SDZ5-Redna-29.ana.seg3...,Hvala lepa. Besedo ima Silva Črnugelj. Priprav...,Besedo ima Silva Črnugelj.,"The word is ""silva blackugelj.""",True,P_Neutral,3.0,3.340636,{'Text_ID': 'ParlaMint-SI-en_2011-06-16-SDZ5-R...


In [24]:
with open('../Datasets/Development_data.jsonl', 'w', encoding='utf-8') as f:
    dev_data.to_json(f, orient='records', lines=True, force_ascii=False)

In [25]:
with open('../Datasets/Test_data.jsonl', 'w', encoding='utf-8') as f:
    test_data.to_json(f, orient='records', lines=True, force_ascii=False)